# Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Load Dataset

In [3]:
df = pd.read_csv("C:\\Users\\user\\Desktop\\IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Data Understanding

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [7]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [13]:
print("Total samples:",len(df))

Total samples: 50000


In [15]:
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [17]:
print(df['review'][0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [19]:
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})

# NLP Preprocessing

In [21]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # lowercase
    text = text.lower()

    # remove numbers
    text = re.sub(r'\d+','',text)

    # remove punctuation
    text = text.translate(str.maketrans('','',string.punctuation))

    # remove special characters
    text = re.sub(r'\W',' ',text)

    # remove extra spaces
    text = re.sub(r'\s+',' ',text).strip()

    # tokenization
    words = text.split()

    # remove stopwords
    words = [word for word in words if word not in stop_words]

    # lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [23]:
df['clean_review'] = df['review'].apply(preprocess_text)

In [25]:
df[['review','clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


# Feature Engineering

In [27]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_review'])

In [29]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_review'])

In [31]:
y = df['sentiment']

# Train Test Split

In [33]:
X_train,X_test,y_train,y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

# Model Building

In [35]:
lr = LogisticRegression()

lr.fit(X_train,y_train)

pred_lr = lr.predict(X_test)

In [37]:
nb = MultinomialNB()

nb.fit(X_train,y_train)

pred_nb = nb.predict(X_test)

In [39]:
dt = DecisionTreeClassifier()

dt.fit(X_train,y_train)

pred_dt = dt.predict(X_test)

# Model Evaluation

In [41]:
def evaluate(y_test,pred):

    print("Accuracy:",
          accuracy_score(y_test,pred))

    print("Precision:",
          precision_score(y_test,pred))

    print("Recall:",
          recall_score(y_test,pred))

    print("F1 Score:",
          f1_score(y_test,pred))

    print(classification_report(y_test,pred))

In [43]:
evaluate(y_test,pred_lr)

Accuracy: 0.8847
Precision: 0.8764044943820225
Recall: 0.8977971819805517
F1 Score: 0.886971865503382
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [45]:
evaluate(y_test,pred_nb)

Accuracy: 0.8517
Precision: 0.8511058451816745
Recall: 0.855328438182179
F1 Score: 0.8532119172523013
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      4961
           1       0.85      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [47]:
evaluate(y_test,pred_dt)

Accuracy: 0.7205
Precision: 0.726027397260274
Recall: 0.715221274062314
F1 Score: 0.7205838248525442
              precision    recall  f1-score   support

           0       0.72      0.73      0.72      4961
           1       0.73      0.72      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.72      0.72      0.72     10000



# Model Comparison Table

In [49]:
results = pd.DataFrame({

'Model':['Logistic Regression',
         'Naive Bayes',
         'Decision Tree'],

'Accuracy':[accuracy_score(y_test,pred_lr),
            accuracy_score(y_test,pred_nb),
            accuracy_score(y_test,pred_dt)],

'F1 Score':[f1_score(y_test,pred_lr),
            f1_score(y_test,pred_nb),
            f1_score(y_test,pred_dt)]

})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.8847,0.886972
1,Naive Bayes,0.8517,0.853212
2,Decision Tree,0.7205,0.720584


# prediction function

In [51]:
def predict_sentiment(text):

    text = preprocess_text(text)

    vector = tfidf.transform([text])

    prediction = lr.predict(vector)

    if prediction[0]==1:
        return "Positive"

    else:
        return "Negative"

In [53]:
predict_sentiment("This movie is amazing")

'Positive'

# Insights

In [ ]:
'''
INSIGHTS

• TF-IDF performed better than Bag of Words
• Logistic Regression gave highest accuracy
• Naive Bayes fastest model
• Decision Tree showed overfitting

Best Model:
Logistic Regression because highest F1 score.

Tradeoff:

| Model         | Advantage      | Disadvantage   |
| ------------- | -------------- | -------------- |
| Logistic      | High accuracy  | Slower         |
| Naive Bayes   | Fast           | Lower accuracy |
| Decision Tree | Easy interpret | Overfitting    |

'''

# Conclusion

In [ ]:
'''

Conclusion:-

This project successfully implemented a complete NLP pipeline including preprocessing, feature engineering, and model training.
Among all models, Logistic Regression performed best with highest accuracy and F1 score. Proper preprocessing significantly improved model performance.

'''